In [2]:
import subprocess
subprocess.run(["pip", "install", "-q", "langgraph", "langchain", "langchain-community",
                "langchain-groq", "wikipedia", "arxiv", "groq"], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'langgraph', 'langchain', 'langchain-community', 'langchain-groq', 'wikipedia', 'arxiv', 'groq'], returncode=0)

In [3]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [4]:
import wikipedia
import arxiv
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [5]:
def wikipedia_tool(topic: str) -> str:
    try:
        summary = wikipedia.summary(topic, sentences=5, auto_suggest=True)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            summary = wikipedia.summary(e.options[0], sentences=5)
            return summary
        except Exception:
            return "Could not retrieve Wikipedia summary."
    except Exception as e:
        return f"Wikipedia lookup failed: {str(e)}"

result = wikipedia_tool("Transformer neural network")
print(result)

Wikipedia lookup failed: Expecting value: line 1 column 1 (char 0)


In [6]:
def arxiv_tool(topic: str) -> str:
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=topic,
            max_results=3,
            sort_by=arxiv.SortCriterion.Relevance
        )
        results = list(client.results(search))
        if not results:
            return "No papers found."
        output = []
        for i, paper in enumerate(results, 1):
            output.append(f"Paper {i}: {paper.title}")
            output.append(f"Authors: {', '.join(str(a) for a in paper.authors[:3])}")
            output.append(f"Abstract: {paper.summary[:400]}...")
            output.append("")
        return "\n".join(output)
    except Exception as e:
        return f"Arxiv lookup failed: {str(e)}"

result = arxiv_tool("large language models")
print(result)

Arxiv lookup failed: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=large+language+models&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)


In [7]:
def llm_knowledge_tool(query: str) -> str:
    response = llm.invoke(query)
    return response.content

result = llm_knowledge_tool("What is the capital of Australia?")
print(result)

The capital of Australia is Canberra.


In [8]:
from typing import TypedDict, Literal, List
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    tool_selected: str
    tool_output: str
    final_response: str
    conversation_history: List[dict]

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [9]:
def router_node(state: AgentState) -> AgentState:
    query = state["query"]
    history = state.get("conversation_history", [])

    history_text = ""
    if history:
        recent = history[-4:]
        history_text = "\n".join([f"{m['role']}: {m['content']}" for m in recent])
        history_text = f"\n\nConversation so far:\n{history_text}"

    prompt = f"""You are a router that selects the best tool for a query.

Available tools:
- wikipedia: for general knowledge, concepts, definitions, history, people, places
- arxiv: for academic research papers, latest studies, scientific findings
- llm: for simple factual questions, general knowledge, out-of-scope or trivial queries
{history_text}

Current query: {query}

Reply with exactly one word: wikipedia, arxiv, or llm"""

    response = llm.invoke(prompt)
    choice = response.content.strip().lower()
    if "arxiv" in choice:
        tool = "arxiv"
    elif "wikipedia" in choice:
        tool = "wikipedia"
    else:
        tool = "llm"

    return {**state, "tool_selected": tool}

In [10]:
def tool_node(state: AgentState) -> AgentState:
    query = state["query"]
    tool = state["tool_selected"]

    if tool == "wikipedia":
        output = wikipedia_tool(query)
    elif tool == "arxiv":
        output = arxiv_tool(query)
    else:
        output = llm_knowledge_tool(query)

    return {**state, "tool_output": output}

In [11]:
def synthesiser_node(state: AgentState) -> AgentState:
    query = state["query"]
    tool = state["tool_selected"]
    tool_output = state["tool_output"]
    history = state.get("conversation_history", [])

    history_text = ""
    if history:
        recent = history[-4:]
        history_text = "\n".join([f"{m['role']}: {m['content']}" for m in recent])
        history_text = f"\n\nConversation history:\n{history_text}"

    if tool == "llm":
        final = tool_output
    else:
        prompt = f"""You are a research assistant. Using the information below, answer the user query clearly and concisely.
{history_text}

User query: {query}

Source ({tool}):
{tool_output}

Provide a well-structured, informative response."""
        response = llm.invoke(prompt)
        final = response.content

    updated_history = history + [
        {"role": "user", "content": query},
        {"role": "assistant", "content": final}
    ]

    return {**state, "final_response": final, "conversation_history": updated_history}

In [12]:
def route_decision(state: AgentState) -> Literal["tool_node", END]:
    return "tool_node"

builder = StateGraph(AgentState)

builder.add_node("router_node", router_node)
builder.add_node("tool_node", tool_node)
builder.add_node("synthesiser_node", synthesiser_node)

builder.set_entry_point("router_node")
builder.add_conditional_edges("router_node", route_decision)
builder.add_edge("tool_node", "synthesiser_node")
builder.add_edge("synthesiser_node", END)

graph = builder.compile()
print("Agent graph compiled successfully.")

Agent graph compiled successfully.


In [13]:
def run_agent(query: str, history: list = None) -> dict:
    if history is None:
        history = []
    initial_state: AgentState = {
        "query": query,
        "tool_selected": "",
        "tool_output": "",
        "final_response": "",
        "conversation_history": history
    }
    result = graph.invoke(initial_state)
    return result

In [14]:
test_queries = [
    "What is Retrieval-Augmented Generation?",
    "What are the latest research papers on large language models?",
    "Who invented the Transformer architecture?",
    "What is the capital of Australia?",
    "What is quantum entanglement?"
]

for query in test_queries:
    print(f"Query: {query}")
    result = run_agent(query)
    print(f"Tool selected: {result['tool_selected']}")
    print(f"Final response:\n{result['final_response']}")
    print()

Query: What is Retrieval-Augmented Generation?
Tool selected: arxiv
Final response:
Retrieval-Augmented Generation (RAG) is a type of natural language processing (NLP) approach that combines the strengths of retrieval-based and generation-based models. 

In traditional generation-based models, the system generates text from scratch based on the input prompt. In contrast, retrieval-based models rely on retrieving relevant information from a database or knowledge graph to answer a query.

RAG models, on the other hand, use a combination of both approaches. They first retrieve relevant information from a database or knowledge graph, and then use this retrieved information as input to generate text. This approach allows RAG models to leverage the strengths of both retrieval and generation, resulting in more accurate and informative text generation.

The key components of a RAG model include:

1. **Retriever**: This module is responsible for retrieving relevant information from a database o

In [15]:
conversation_history = []

multi_turn_queries = [
    "What is a Transformer?",
    "How is it different from an RNN?",
    "Which one would you use for text generation and why?"
]

for query in multi_turn_queries:
    print(f"User: {query}")
    result = run_agent(query, history=conversation_history)
    conversation_history = result["conversation_history"]
    print(f"Tool selected: {result['tool_selected']}")
    print(f"Assistant: {result['final_response']}")
    print()

User: What is a Transformer?
Tool selected: wikipedia
Assistant: Unfortunately, the Wikipedia lookup failed to provide information on the topic. However, I can provide a general overview of what a Transformer is based on common knowledge.

A Transformer is a type of neural network architecture introduced in 2017 by Vaswani et al. in the paper "Attention Is All You Need." It is primarily used for natural language processing (NLP) tasks, such as language translation, text generation, and text classification.

The Transformer model relies on self-attention mechanisms to weigh the importance of different input elements, allowing it to handle sequential data more efficiently than traditional recurrent neural networks (RNNs). This architecture has become a standard component in many state-of-the-art NLP models, including BERT, RoBERTa, and others.

If you're looking for more detailed information on the Transformer architecture, I recommend searching for academic papers or online resources th

In [16]:
evaluation_queries = [
    {
        "query": "What is Retrieval-Augmented Generation?",
        "expected_keywords": ["retrieval", "generation", "knowledge", "document", "llm"],
        "expected_tool": "wikipedia"
    },
    {
        "query": "What are the latest research papers on large language models?",
        "expected_keywords": ["paper", "model", "language", "abstract", "authors"],
        "expected_tool": "arxiv"
    },
    {
        "query": "Who invented the Transformer architecture?",
        "expected_keywords": ["vaswani", "attention", "google", "2017", "transformer"],
        "expected_tool": "wikipedia"
    },
    {
        "query": "What is the capital of Australia?",
        "expected_keywords": ["canberra", "australia", "capital"],
        "expected_tool": "llm"
    },
    {
        "query": "What is quantum entanglement?",
        "expected_keywords": ["quantum", "entanglement", "particle", "state"],
        "expected_tool": "wikipedia"
    }
]

def evaluate_response(query: str, response: str, tool_selected: str,
                       expected_keywords: list, expected_tool: str) -> dict:
    response_lower = response.lower()

    matched = [kw for kw in expected_keywords if kw.lower() in response_lower]
    accuracy = len(matched) / len(expected_keywords)

    query_words = set(query.lower().split())
    stop_words = {"what", "is", "the", "a", "an", "of", "for", "are", "who", "how", "latest"}
    key_query_words = query_words - stop_words
    relevance_matches = sum(1 for w in key_query_words if w in response_lower)
    relevance = min(relevance_matches / max(len(key_query_words), 1), 1.0)

    groundedness = 1.0 if tool_selected == expected_tool else 0.5

    return {
        "accuracy": round(accuracy, 2),
        "relevance": round(relevance, 2),
        "groundedness": round(groundedness, 2),
        "tool_correct": tool_selected == expected_tool,
        "matched_keywords": matched
    }

print("Evaluation Results")


all_scores = []
for item in evaluation_queries:
    result = run_agent(item["query"])
    scores = evaluate_response(
        query=item["query"],
        response=result["final_response"],
        tool_selected=result["tool_selected"],
        expected_keywords=item["expected_keywords"],
        expected_tool=item["expected_tool"]
    )
    all_scores.append(scores)

    print(f"Query: {item['query']}")
    print(f"Tool selected: {result['tool_selected']} (expected: {item['expected_tool']}) - Correct: {scores['tool_correct']}")
    print(f"Accuracy:      {scores['accuracy']}")
    print(f"Relevance:     {scores['relevance']}")
    print(f"Groundedness:  {scores['groundedness']}")
    print(f"Matched keywords: {scores['matched_keywords']}")
    print()

avg_accuracy = sum(s["accuracy"] for s in all_scores) / len(all_scores)
avg_relevance = sum(s["relevance"] for s in all_scores) / len(all_scores)
avg_groundedness = sum(s["groundedness"] for s in all_scores) / len(all_scores)
tool_accuracy = sum(1 for s in all_scores if s["tool_correct"]) / len(all_scores)

print("Average Scores Across All Queries")
print(f"Average Accuracy:      {round(avg_accuracy, 2)}")
print(f"Average Relevance:     {round(avg_relevance, 2)}")
print(f"Average Groundedness:  {round(avg_groundedness, 2)}")
print(f"Tool Selection Accuracy: {round(tool_accuracy, 2)}")

Evaluation Results
Query: What is Retrieval-Augmented Generation?
Tool selected: arxiv (expected: wikipedia) - Correct: False
Accuracy:      0.6
Relevance:     0.5
Groundedness:  0.5
Matched keywords: ['retrieval', 'generation', 'knowledge']

Query: What are the latest research papers on large language models?
Tool selected: arxiv (expected: arxiv) - Correct: True
Accuracy:      0.6
Relevance:     0.83
Groundedness:  1.0
Matched keywords: ['paper', 'model', 'language']

Query: Who invented the Transformer architecture?
Tool selected: llm (expected: wikipedia) - Correct: False
Accuracy:      1.0
Relevance:     0.33
Groundedness:  0.5
Matched keywords: ['vaswani', 'attention', 'google', '2017', 'transformer']

Query: What is the capital of Australia?
Tool selected: llm (expected: llm) - Correct: True
Accuracy:      1.0
Relevance:     0.5
Groundedness:  1.0
Matched keywords: ['canberra', 'australia', 'capital']

Query: What is quantum entanglement?
Tool selected: wikipedia (expected: wiki